In [1]:
!pip install "C:/Users/ASUS/Downloads/llama_cpp_python-0.3.2-cp312-cp312-win_amd64.whl"

Defaulting to user installation because normal site-packages is not writeable
Processing c:\users\asus\downloads\llama_cpp_python-0.3.2-cp312-cp312-win_amd64.whl
llama-cpp-python is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [34]:
from llama_cpp import Llama

llm = Llama(
    model_path=r"D:\infosys internship ai knowlege grpah builder\models\Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    n_ctx=2048,
    n_threads=6,
    n_gpu_layers=0,
    verbose=False
)

print("Llama model loaded successfully")

llama_new_context_with_model: n_ctx_per_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Llama model loaded successfully


In [35]:
result = llm("Say hello in one sentence.")
print(result["choices"][0]["text"])

 (Note: I will write the sentence with the name "Sally" in


In [36]:
def build_prompt(record):
    return f"""
Extract entities and relationships from this record in JSON only.

Record:
{record}

Return strictly JSON:
{{
  "entities": [],
  "relationships": []
}}
"""

In [37]:
import pandas as pd

df = pd.read_csv("fortune1000_cleaned.csv")  # or the cleaned file you saved

In [38]:
important_cols = [
    "Company", "CEO", "Sector", "Industry",
    "HeadquartersCity", "HeadquartersState",
    "Country", "CompanyType"
]

test_row = df[important_cols].iloc[0]

prompt = build_prompt(test_row.to_dict())
result = llm(prompt)
print(result["choices"][0]["text"])

``` 

Here is the record in a JSON object that includes both entities and relationships


In [39]:
import pandas as pd
import json

# Load your cleaned dataset
df = pd.read_csv("fortune1000_cleaned.csv")

important_cols = [
    "Company", "CEO", "Sector", "Industry",
    "HeadquartersCity", "HeadquartersState",
    "Country", "CompanyType"
]

entities = []
relationships = []

# --- Helper functions ---
def make_id(text):
    return text.strip().replace(" ", "_").replace("/", "_").replace("&", "_")

def add_entity(e_type, name):
    eid = f"{e_type}_{make_id(name)}"
    return {"id": eid, "type": e_type, "name": name}

def add_relationship(source, rel_type, target):
    return {"source": source, "type": rel_type, "target": target}

# --- Main Loop ---
for _, row in df[important_cols].iterrows():

    company = add_entity("Company", row["Company"])
    ceo = add_entity("CEO", row["CEO"])
    sector = add_entity("Sector", row["Sector"])
    industry = add_entity("Industry", row["Industry"])
    city = add_entity("City", row["HeadquartersCity"])
    state = add_entity("State", row["HeadquartersState"])
    country = add_entity("Country", row["Country"])

    entities += [company, ceo, sector, industry, city, state, country]

    # Relationships
    relationships += [
        add_relationship(company["id"], "HAS_CEO", ceo["id"]),
        add_relationship(company["id"], "IN_SECTOR", sector["id"]),
        add_relationship(company["id"], "IN_INDUSTRY", industry["id"]),
        add_relationship(company["id"], "LOCATED_IN", city["id"]),
        add_relationship(city["id"], "IN_STATE", state["id"]),
        add_relationship(state["id"], "IN_COUNTRY", country["id"])
    ]

# --- Build final KG file ---
kg = {"entities": entities, "relationships": relationships}

with open("kg_fortune1000.json", "w") as f:
    json.dump(kg, f, indent=4)

print("KG built successfully!")

KG built successfully!


In [40]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)

def test_connection(tx):
    result = tx.run("RETURN 'Neo4j connection successful!' AS message")
    print(result.single()["message"])

with driver.session() as session:
    session.execute_read(test_connection)

driver.close()

Neo4j connection successful!


In [9]:
import json

with open("kg_fortune1000.json", "r") as f:
    kg = json.load(f)

print("Entities:", len(kg["entities"]))
print("Relationships:", len(kg["relationships"]))

Entities: 7000
Relationships: 6000


In [41]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)

def insert_entity(tx, entity):
    tx.run(
        """
        MERGE (e:Entity {id: $id})
        SET e.type = $type,
            e.name = $name
        """,
        id=entity["id"],
        type=entity["type"],
        name=entity["name"]
    )

with driver.session() as session:
    for ent in kg["entities"]:
        session.execute_write(insert_entity, ent)

driver.close()

print("Entities inserted successfully!")

Entities inserted successfully!


In [42]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)

def insert_relationship(tx, rel):
    tx.run(
        """
        MATCH (a:Entity {id: $source})
        MATCH (b:Entity {id: $target})
        MERGE (a)-[:RELATION {type:$type}]->(b)
        """,
        source=rel["source"],
        target=rel["target"],
        type=rel["type"]
    )

with driver.session() as session:
    for rel in kg["relationships"]:
        session.execute_write(insert_relationship, rel)

driver.close()

print("Relationships inserted successfully!")

Relationships inserted successfully!


In [62]:
prompt = f"""
You are an information extraction system.

Extract entities and relationships from the company record.

Return ONLY valid JSON.
Do not add explanations.
Do not repeat the JSON.

Schema:

{{
 "entities":[
  {{"type":"Company","name":""}},
  {{"type":"CEO","name":""}},
  {{"type":"Sector","name":""}},
  {{"type":"Industry","name":""}},
  {{"type":"City","name":""}},
  {{"type":"State","name":""}},
  {{"type":"Country","name":""}}
 ],
 "relationships":[
  {{"source":"","relation":"HAS_CEO","target":""}},
  {{"source":"","relation":"IN_SECTOR","target":""}},
  {{"source":"","relation":"IN_INDUSTRY","target":""}},
  {{"source":"","relation":"LOCATED_IN","target":""}},
  {{"source":"","relation":"IN_STATE","target":""}},
  {{"source":"","relation":"IN_COUNTRY","target":""}}
 ]
}}

Company Record:

Company: {sample['Company']}
CEO: {sample['CEO']}
Sector: {sample['Sector']}
Industry: {sample['Industry']}
City: {sample['HeadquartersCity']}
State: {sample['HeadquartersState']}
Country: {sample['Country']}
"""

In [63]:
import pandas as pd

df = pd.read_csv("fortune1000_cleaned.csv")

important_cols = [
    "Company","CEO","Sector","Industry",
    "HeadquartersCity","HeadquartersState","Country"
]

sample = df[important_cols].iloc[0]

sample


Company                            Walmart
CEO                    C. Douglas Mcmillon
Sector                           Retailing
Industry             General Merchandisers
HeadquartersCity               Bentonville
HeadquartersState                 Arkansas
Country                               U.S.
Name: 0, dtype: object

In [64]:
prompt = build_prompt(sample.to_dict())
print(prompt)


Fill the JSON template using the company record below.

Record:
Company: Walmart
CEO: C. Douglas Mcmillon
Sector: Retailing
Industry: General Merchandisers
City: Bentonville
State: Arkansas
Country: U.S.

Return ONLY the filled JSON.

{
 "entities":[
  {"type":"Company","name":"Walmart"},
  {"type":"CEO","name":"C. Douglas Mcmillon"},
  {"type":"Sector","name":"Retailing"},
  {"type":"Industry","name":"General Merchandisers"},
  {"type":"City","name":"Bentonville"},
  {"type":"State","name":"Arkansas"},
  {"type":"Country","name":"U.S."}
 ],
 "relationships":[
  {"source":"Walmart","relation":"HAS_CEO","target":"C. Douglas Mcmillon"},
  {"source":"Walmart","relation":"IN_SECTOR","target":"Retailing"},
  {"source":"Walmart","relation":"IN_INDUSTRY","target":"General Merchandisers"},
  {"source":"Walmart","relation":"LOCATED_IN","target":"Bentonville"}
 ]
}



In [65]:
result = llm(
    prompt,
    max_tokens=300,
    stop=["\n\n"]   # stops after JSON
)

print(result["choices"][0]["text"])

{
 "entities":[
  {"type":"Company","name":"Walmart"},
  {"type":"CEO","name":"C. Douglas Mcmillon"},
  {"type":"Sector","name":"Retailing"},
  {"type":"Industry","name":"General Merchandisers"},
  {"type":"City","name":"Bentonville"},
  {"type":"State","name":"Arkansas"},
  {"type":"Country","name":"U.S."}
 ],
 "relationships":[
  {"source":"Walmart","relation":"HAS_CEO","target":"C. Douglas Mcmillon"},
  {"source":"Walmart","relation":"IN_SECTOR","target":"Retailing"},
  {"source":"Walmart","relation":"IN_INDUSTRY","target":"General Merchandisers"},
  {"source":"Walmart","relation":"LOCATED_IN","target":"Bentonville"}
 ]
} 


In [66]:
import json

data = json.loads(result["choices"][0]["text"])

entities = data["entities"]
relationships = data["relationships"]

print("Entities:", entities)
print("Relationships:", relationships)

Entities: [{'type': 'Company', 'name': 'Walmart'}, {'type': 'CEO', 'name': 'C. Douglas Mcmillon'}, {'type': 'Sector', 'name': 'Retailing'}, {'type': 'Industry', 'name': 'General Merchandisers'}, {'type': 'City', 'name': 'Bentonville'}, {'type': 'State', 'name': 'Arkansas'}, {'type': 'Country', 'name': 'U.S.'}]
Relationships: [{'source': 'Walmart', 'relation': 'HAS_CEO', 'target': 'C. Douglas Mcmillon'}, {'source': 'Walmart', 'relation': 'IN_SECTOR', 'target': 'Retailing'}, {'source': 'Walmart', 'relation': 'IN_INDUSTRY', 'target': 'General Merchandisers'}, {'source': 'Walmart', 'relation': 'LOCATED_IN', 'target': 'Bentonville'}]


In [69]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)

def insert_entity(tx, entity):
    tx.run(
        """
        MERGE (e:Entity {name:$name})
        SET e.type=$type
        """,
        name=entity["name"],
        type=entity["type"]
    )

def insert_relationship(tx, rel):
    tx.run(
        """
        MATCH (a:Entity {name:$source})
        MATCH (b:Entity {name:$target})
        MERGE (a)-[:RELATION {type:$relation}]->(b)
        """,
        source=rel["source"],
        target=rel["target"],
        relation=rel["relation"]
    )

with driver.session() as session:
    for e in entities:
        session.execute_write(insert_entity, e)

    for r in relationships:
        session.execute_write(insert_relationship, r)

driver.close()

print("Graph updated from LLM output!")

Graph updated from LLM output!
